# Notebook 04-CIC — SHAP on CIC-IDS2017 (streamlined)

**Project:** Calibrated and Stability-Aware Explainable Intrusion Detection
**Author:** Md Anas Biswas, University of Portsmouth
**Dataset:** CIC-IDS2017 (200K subsample)

## Streamlined methodology (vs NSL-KDD Notebook 04)

- **RF subsample:** 1000 stratified eval samples (vs 2433 for NSL-KDD) — CIC has fewer features so SHAP is faster per sample, and we just need top-feature rankings to compare with NSL-KDD
- **DNN background:** 200 stratified train samples (vs 300 for NSL-KDD)
- **XGBoost:** Same native `pred_contribs` workaround as NSL-KDD
- **Per-class rankings:** Skipped (NSL-KDD already established cross-class patterns; CIC just needs aggregate)

## Output

```
shap_values/cic_ids2017/
├── <name>_shap.npy  ×6  (SHAP arrays per model)
├── rf_subsample_idx.npy
├── feature_importance_rankings.json (top-10 per model)
results/figures/cic_top10_features.png
results/tables/cic_top10_features.csv
```

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

!git pull
print(f'✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
Already up to date.
✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
!pip install -q shap==0.47.2 2>&1 | tail -2
print('✓ SHAP ready')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.6 MB/s eta 0:00:00
✓ SHAP ready


In [3]:
import numpy as np
import pandas as pd
import json, pickle, time
from pathlib import Path

import shap
import xgboost as xgb
import torch
import torch.nn as nn

import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'SHAP: {shap.__version__}')

Device: cuda
SHAP: 0.47.2


In [4]:
PROCESSED = Path(REPO) / 'data' / 'processed' / 'cic_ids2017'
MODELS_DIR = Path(REPO) / 'models' / 'cic_ids2017'
CALIB_DIR = Path(REPO) / 'calibrators' / 'cic_ids2017'
SHAP_DIR = Path(REPO) / 'shap_values' / 'cic_ids2017'
SHAP_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = Path(REPO) / 'results' / 'figures'
TABLES_DIR = Path(REPO) / 'results' / 'tables'

X_train = np.load(PROCESSED / 'X_train.npy')
X_test = np.load(PROCESSED / 'X_test.npy')
y_train_5 = np.load(PROCESSED / 'y_train_5class.npy')
y_test_b = np.load(PROCESSED / 'y_test_binary.npy')
y_test_5 = np.load(PROCESSED / 'y_test_5class.npy')
idx_eval = np.load(CALIB_DIR / 'idx_eval.npy')

X_eval = X_test[idx_eval]
y_eval_b = y_test_b[idx_eval]
y_eval_5 = y_test_5[idx_eval]

with open(PROCESSED / 'feature_names.json') as f:
    FEATURE_NAMES = json.load(f)

print(f'X_eval: {X_eval.shape}')
print(f'Features: {len(FEATURE_NAMES)}')

X_eval: (20004, 78)
Features: 78


---
## Step 1 — RF subsample of 1000 stratified eval samples

In [5]:
RF_SUB_SIZE = 1000
rng = np.random.RandomState(SEED)

# Stratified by 5-class
rf_idx = []
per_class = RF_SUB_SIZE // 5
for c in range(5):
    pool = np.where(y_eval_5 == c)[0]
    take = min(per_class, len(pool))
    rf_idx.extend(rng.choice(pool, take, replace=False).tolist())
rf_idx = np.array(rf_idx)
X_rf_sub = X_eval[rf_idx]

np.save(SHAP_DIR / 'rf_subsample_idx.npy', rf_idx)
print(f'RF subsample: {X_rf_sub.shape}')
print(f'Class distribution: {np.bincount(y_eval_5[rf_idx])}')

RF subsample: (715, 78)
Class distribution: [200 200 200 112   3]


---
## Step 2 — DNN background (200 stratified train samples)

In [6]:
BG_SIZE = 200
rng_bg = np.random.RandomState(SEED + 1)
bg_idx = []
for c in range(5):
    pool = np.where(y_train_5 == c)[0]
    take = min(BG_SIZE // 5, len(pool))
    bg_idx.extend(rng_bg.choice(pool, take, replace=False).tolist())
bg_idx = np.array(bg_idx)
X_background = X_train[bg_idx]

print(f'DNN background: {X_background.shape}')

DNN background: (189, 78)


---
## Step 3 — Model loading helpers + DNN MLP class

In [7]:
class MLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(256, 128, 64, 32), dropout=0.3):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def load_model(name):
    pkl_path = MODELS_DIR / f'{name}.pkl'
    pt_path = MODELS_DIR / f'{name}.pt'
    if pkl_path.exists():
        with open(pkl_path, 'rb') as f:
            return ('sklearn', pickle.load(f))
    state = torch.load(pt_path, map_location=DEVICE, weights_only=False)
    m = MLP(state['in_dim'], state['n_classes'],
            hidden=tuple(state['hidden']), dropout=state['dropout']).to(DEVICE)
    m.load_state_dict(state['state_dict']); m.eval()
    return ('torch', m)

CANONICAL = {
    'rf_binary_cw':      {'target': 'binary', 'kind': 'tree', 'lib': 'rf'},
    'xgb_binary_cw':     {'target': 'binary', 'kind': 'tree', 'lib': 'xgb'},
    'dnn_binary_cw':     {'target': 'binary', 'kind': 'dnn'},
    'rf_5class_smote':   {'target': '5class', 'kind': 'tree', 'lib': 'rf'},
    'xgb_5class_smote':  {'target': '5class', 'kind': 'tree', 'lib': 'xgb'},
    'dnn_5class_smote':  {'target': '5class', 'kind': 'dnn'},
}

MODELS = {n: load_model(n) for n in CANONICAL}
for n in MODELS:
    print(f'  ✓ {n}')

  ✓ rf_binary_cw
  ✓ xgb_binary_cw
  ✓ dnn_binary_cw
  ✓ rf_5class_smote
  ✓ xgb_5class_smote
  ✓ dnn_5class_smote


---
## Step 4 — Compute SHAP per model

Expected runtime: ~50-70 min total.

In [ ]:
SHAP_ARRAYS = {}

for name, info in CANONICAL.items():
    print(f'\n=== {name} ===')
    t0 = time.time()
    kind, model = MODELS[name]

    # Decide input set: RF uses subsample, others use full eval
    if info.get('lib') == 'rf':
        X_input = X_rf_sub
    else:
        X_input = X_eval  # full eval (~20k samples)

    if info['kind'] == 'tree':
        if info.get('lib') == 'xgb':
            # Native pred_contribs (bypass shap.TreeExplainer bug)
            booster = model.get_booster()
            dmat = xgb.DMatrix(X_input)
            raw = booster.predict(dmat, pred_contribs=True)
            if raw.ndim == 2:
                # Binary: (n, f+1) → drop bias, stack to (n, f, 2)
                arr = raw[:, :-1]
                arr = np.stack([-arr, arr], axis=-1)
            else:
                # Multi-class: (n, n_classes, f+1) → (n, f, n_classes)
                arr = raw[:, :, :-1]
                arr = np.transpose(arr, (0, 2, 1))
        else:
            # RF via TreeExplainer
            explainer = shap.TreeExplainer(model)
            raw = explainer.shap_values(X_input)
            arr = np.stack(raw, axis=-1) if isinstance(raw, list) else np.asarray(raw)
    else:
        # DNN — DeepExplainer
        bg_t = torch.tensor(X_background, dtype=torch.float32).to(DEVICE)
        X_t = torch.tensor(X_input, dtype=torch.float32).to(DEVICE)
        explainer = shap.DeepExplainer(model, bg_t)
        raw = explainer.shap_values(X_t, check_additivity=False)
        arr = np.stack(raw, axis=-1) if isinstance(raw, list) else np.asarray(raw)

    arr = arr.astype(np.float32)
    SHAP_ARRAYS[name] = arr
    np.save(SHAP_DIR / f'{name}_shap.npy', arr)
    print(f'  Shape: {arr.shape}  ({time.time()-t0:.1f}s)')


=== rf_binary_cw ===
  Shape: (715, 78, 2)  (91.6s)

=== xgb_binary_cw ===
  Shape: (20004, 78, 2)  (13.8s)

=== dnn_binary_cw ===
  Shape: (20004, 78, 2)  (434.0s)

=== rf_5class_smote ===
  Shape: (715, 78, 5)  (137.4s)

=== xgb_5class_smote ===
  Shape: (20004, 78, 5)  (52.9s)

=== dnn_5class_smote ===


---
## Step 5 — Compute top-10 feature rankings

In [ ]:
def get_global_top_features(shap_arr, k=10):
    '''Mean absolute SHAP per feature (aggregated across classes for multi-class).'''
    if shap_arr.ndim == 3:
        # (n, f, n_classes) → aggregate across classes
        imp = np.abs(shap_arr).mean(axis=(0, 2))
    else:
        imp = np.abs(shap_arr).mean(axis=0)
    top_idx = np.argsort(-imp)[:k]
    return [(FEATURE_NAMES[i], float(imp[i])) for i in top_idx]

rankings = {}
print('Top-10 features per model (CIC-IDS2017):\n')
for name in CANONICAL:
    top = get_global_top_features(SHAP_ARRAYS[name], k=10)
    rankings[name] = top
    print(f'\n--- {name} ---')
    for i, (feat, val) in enumerate(top):
        print(f'  {i+1:2d}. {feat:<30} {val:.4f}')

with open(SHAP_DIR / 'feature_importance_rankings.json', 'w') as f:
    json.dump({k: [[f, v] for f, v in v] for k, v in rankings.items()}, f, indent=2)

---
## Step 6 — Top-10 overlap analysis

In [ ]:
def get_top_set(rankings_dict, name, k=10):
    return set(f for f, _ in rankings_dict[name][:k])

print('Cross-architecture top-10 overlap (CIC-IDS2017):\n')
for target in ['binary', '5class']:
    names = [n for n in CANONICAL if CANONICAL[n]['target'] == target]
    rf_set = get_top_set(rankings, names[0])
    xgb_set = get_top_set(rankings, names[1])
    dnn_set = get_top_set(rankings, names[2])

    rf_xgb = rf_set & xgb_set
    rf_dnn = rf_set & dnn_set
    xgb_dnn = xgb_set & dnn_set
    all3 = rf_set & xgb_set & dnn_set

    print(f'{target}:')
    print(f'  RF ∩ XGB: {len(rf_xgb)}/10  — {sorted(rf_xgb)}')
    print(f'  RF ∩ DNN: {len(rf_dnn)}/10')
    print(f'  XGB ∩ DNN: {len(xgb_dnn)}/10')
    print(f'  All 3:    {len(all3)}/10  — {sorted(all3)}')
    print()

In [ ]:
# Top-10 features table
rows = []
for name in CANONICAL:
    for rank, (feat, val) in enumerate(rankings[name], 1):
        rows.append({'Model': name, 'Rank': rank, 'Feature': feat, 'mean_abs_SHAP': val})
df_top = pd.DataFrame(rows)
df_top.to_csv(TABLES_DIR / 'cic_top10_features.csv', index=False)
print(f'✓ Saved top-10 features table')

---
## Step 7 — Visualisations (3x2 grid of top-10 SHAP)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
models_order = list(CANONICAL.keys())
for ax, name in zip(axes.flatten(), models_order):
    top = rankings[name]
    feats = [f for f, _ in top]
    vals = [v for _, v in top]
    ax.barh(range(len(feats)), vals[::-1], color='#4C72B0')
    ax.set_yticks(range(len(feats)))
    ax.set_yticklabels(feats[::-1], fontsize=8)
    ax.set_xlabel('mean |SHAP|')
    ax.set_title(name, fontsize=10)
    ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'cic_top10_features.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Commit
os.chdir(REPO)
!git add notebooks/04_cic_shap.ipynb
!git add results/
!git add shap_values/cic_ids2017/feature_importance_rankings.json
!git add shap_values/cic_ids2017/rf_subsample_idx.npy
!git status --short
!git commit -m 'Notebook 04-CIC: SHAP for 6 canonical models on CIC-IDS2017'
!git push

## Summary

- ✓ SHAP computed for all 6 models on CIC-IDS2017
- ✓ Top-10 rankings extracted and saved
- ✓ Cross-architecture overlap analysed
- ✓ Visualisation generated

**Next:** Notebook 05-CIC (stability tests).
